# Non-Abelian Groups as Uniformly Automatic Classes

For abelian groups, addition is automatic because collecting a product in a polycyclic
presentation never creates error terms. Once generators stop commuting, every collection
step conjugates tail generators past head generators — and whether the class stays
uniformly automatic depends on whether a synchronous automaton can absorb those
corrections:

* **advice-definable constants** (like $s^2 = r^w$) are free — write $w$ on the advice tape;
* **inversion** is free — it is addition with permuted arguments;
* **shifts at advice-marked positions** are free — multiplying by $2^{k-2} \pm 1$ mod $2^{k-1}$
  only touches the top digit;
* **genuine bilinearity between two unbounded values is fatal** — it embeds integer
  multiplication, which no automaton recognizes.

`autstr.groups` implements two classes sitting right on the good side of this line.

## Groups with a cyclic subgroup of index ≤ 2

These are classified into six families over the cyclic part $\mathbb{Z}_n$:

| family | group | conjugation $u$ | twist square $w$ |
|---|---|---|---|
| `abelian` | $\mathbb{Z}_2 \times \mathbb{Z}_n$ | $1$ | $0$ |
| `cyclic` | $C_{2n}$ | $1$ | $1$ |
| `dihedral` | $D_n$ | $-1$ | $0$ |
| `dicyclic` | $\mathrm{Dic}_n$ (e.g. $Q_8$) | $-1$ | $n/2$ |
| `semidihedral` | $SD_{2n}$ | $n/2-1$ | $0$ |
| `modular` | $M_{2n}$ | $n/2+1$ | $0$ |

The advice is one family symbol followed by the binary digits of $n$; an element
$r^a s^e$ is a twist bit followed by the digits of $a$. The multiplication relation is
**defined first-order** from tiny primitives (modular addition, negation, the
$n/2$-shift, parity) via `UniformlyAutomaticClass.define` — the uniform analog of the
Büchi-arithmetic bootstrap.

In [ ]:
from autstr.groups import IndexTwoCyclicGroups

%time idx2 = IndexTwoCyclicGroups()
print('multiplication automaton:', idx2.cls.class_automata['M'].num_states, 'states')

In [ ]:
for name, advice in [('D_4', idx2.dihedral(4)), ('Q_8', idx2.dicyclic(4)),
                     ('SD_16', idx2.semidihedral(8)), ('M_16', idx2.modular(8)),
                     ('C_6', idx2.cyclic(3)), ('Z_2 x Z_5', idx2.abelian(5))]:
    print(f'{name:>9}: advice = {advice}')

### Multiplying in the quaternion group

$Q_8$ is `dicyclic(4)`: with $i = r$ and $j = s$, the element $(e, a)$ is $r^a s^e$.
The 181-state automaton decides every product — here $i \cdot j$, and the famous
$j^2 = r^2 = -1$:

In [ ]:
q8 = idx2.dicyclic(4)
i, j = (0, 1), (1, 0)

ij = idx2.multiply(q8, i, j)      # reference group law
jj = idx2.multiply(q8, j, j)
print('i*j =', ij, ' j*j =', jj, ' (r^2 = -1)')
print('automaton agrees on i*j:  ', idx2.check('M(x,y,z)', q8, x=i, y=j, z=ij))
print('automaton accepts i*j = 1: ', idx2.check('M(x,y,z)', q8, x=i, y=j, z=(0, 0)))

### One automaton, the whole zoo

Compile *is non-abelian* once; every member group is then classified by running its
advice word. Note $D_2$ — the Klein four-group — is correctly recognized as abelian:

In [ ]:
nonabelian = 'exists x.(exists y.(exists z.(M(x,y,z) and (not M(y,x,z)))))'
dfa, tapes = idx2.evaluate(nonabelian)
assert tapes == ['advice']

zoo = [('Z_2 x Z_9', idx2.abelian(9)), ('C_14', idx2.cyclic(7)),
       ('D_2 (Klein)', idx2.dihedral(2)), ('D_3', idx2.dihedral(3)),
       ('D_16', idx2.dihedral(16)), ('Q_8', idx2.dicyclic(4)),
       ('Dic_12', idx2.dicyclic(12)), ('SD_32', idx2.semidihedral(16)),
       ('M_32', idx2.modular(16))]
for name, advice in zoo:
    print(f'{name:>12}: non-abelian = {dfa.accepts([(s,) for s in advice])}')

A subtler invariant: $Q_8$ has a *unique* involution (the central $-1$), while $D_4$
has several — a first-order sentence separates them:

In [ ]:
two_involutions = ('exists x.(exists y.((not Eq(x,y)) and (not Eq(x,o)) and '
                   '(not Eq(y,o)) and M(x,x,o) and M(y,y,o)))')
print('Q_8 has two distinct involutions:', idx2.check(two_involutions, idx2.dicyclic(4), o=(0, 0)))
print('D_4 has two distinct involutions:', idx2.check(two_involutions, idx2.dihedral(4), o=(0, 0)))

## Extraspecial $p$-groups: nilpotency class 2, the possible half

For a **fixed** prime $p$, the Heisenberg-type group of order $p^{1+2n}$ has elements
$(c, \mathbf{a}, \mathbf{b})$ with multiplication

$$(c,\mathbf a,\mathbf b)(c',\mathbf a',\mathbf b') = (c + c' + \langle \mathbf a, \mathbf b'\rangle,\ \mathbf a + \mathbf a',\ \mathbf b + \mathbf b').$$

The commutator correction $\langle \mathbf a, \mathbf b'\rangle$ is bilinear — but it
pairs digits *positionwise*, so it is just a running sum mod $p$ in the automaton state.
Contrast: the same group over $\mathbb{Z}_n$ with *growing modulus* interprets modular
multiplication of unbounded numbers via commutators, so that class is **provably not**
uniformly automatic. The dividing line of nilpotency class 2 runs between growing rank
(fine) and growing modulus (fatal).

In [ ]:
from autstr.groups import ExtraspecialGroups

%time heis3 = ExtraspecialGroups(3)
print('M automaton:', heis3.cls.class_automata['M'].num_states, 'states')

g = (1, (2, 0), (1, 1))
h = (0, (1, 1), (2, 0))
gh = heis3.multiply(g, h)
print('g   =', heis3.encode(g, 2))
print('g*h =', gh, '->', heis3.check('M(x,y,z)', 2, x=g, y=h, z=gh))

In [ ]:
# The center is exactly the commuting elements — and one automaton
# detects non-commutativity for every rank
commutes_with_all = 'all y.(all z.((not M(x,y,z)) or M(y,x,z)))'
print('central (1,0,0) commutes with all:', heis3.check(commutes_with_all, 1, x=(1, (0,), (0,))))
print('generator commutes with all:      ', heis3.check(commutes_with_all, 1, x=(0, (1,), (0,))))

nonabelian = 'exists x.(exists y.(exists z.(M(x,y,z) and (not M(y,x,z)))))'
dfa, _ = heis3.evaluate(nonabelian)
for n in range(4):
    print(f'rank {n} (order 3^{1 + 2 * n}): non-abelian =',
          dfa.accepts([(s,) for s in heis3.advice(n)]))

In [ ]:
# Exponent p: for odd p every element has order dividing p ...
exponent3 = 'all x.(exists a.(exists b.(M(x,x,a) and M(a,x,b) and M(b,b,b))))'
print('exponent 3 (p=3, rank 2):', heis3.check(exponent3, 2))

# ... while p = 2 extraspecial groups contain elements of order 4
heis2 = ExtraspecialGroups(2)
squares_trivial = 'all x.(exists a.(M(x,x,a) and M(a,a,a)))'
print('exponent 2 (p=2, rank 0):', heis2.check(squares_trivial, 0))
print('exponent 2 (p=2, rank 1):', heis2.check(squares_trivial, 1), ' (D_4-type: order-4 elements)')

---

**The bigger picture.** Together with `algebra_showcase.ipynb` (finite Boolean algebras,
finite abelian groups, $\mathbb{Z}[1/p]$) and `treedepth_graphs.ipynb` (bounded
tree-depth and pathwidth graphs with full MSO), this notebook maps out the practical
frontier of uniform automaticity: abelian-by-bounded groups with tame actions and
fixed-exponent central extensions are in; unbounded bilinearity — and with it the
general finite nilpotent group of class 2 — is out. All of it is built on
`autstr.uniform.UniformlyAutomaticClass`, whose `define()` turns first-order
definitions over small primitive automata into new class relations.